# 每日推薦 + 離場追蹤 → LINE

**用法**：Main 跑完後執行（Main 最後 `%run _DailySignal.ipynb`），或單獨 Restart & Run All。

| 區塊 | 內容 |
|---|---|
| 1. 設定 | 路徑、離場參數、是否發 LINE |
| 2. 載入 | 讀 `Result/DailyFiles`、推薦紀錄、處理進度 |
| 3. 持倉更新 | 補進場價（推薦隔日開盤）、檢查離場條件 |
| 4. 新推薦 | 當日 `推薦進場` 全部收進推薦紀錄 |
| 5. 存檔 | `Result/recommend/推薦紀錄.csv`、`state.json` |
| 5b. 網頁資料 | `Result/recommend/web/` 依推薦月份拆 JSON，給 `recommend.html` 讀 |
| 6. LINE | 新推薦 / 離場訊號 / 追蹤中，每列點擊開 Yahoo 股市 |

**推薦紀錄**獨立存放，不受 Main 清空 `DailyFiles` 影響。
**處理進度**記在 `state.json`：只處理比上次更新的日期，同一天重跑不會重複推薦、也不會重複發 LINE。
漏跑幾天時（DailyFiles 約保留 20 天），會依日期順序補處理。

### 離場規則（收盤判斷，任一成立即離場）
1. **停損**：收盤 < 停損價。停損價 = 價起回測取「突破線 × 0.97」、量起後量縮守穩取「量起 K 低」，且最多虧 8%
2. **跌破 MA20**
3. **移動停利**：曾獲利 ≥ 15% 後，自最高收盤回落 10%
4. **時間出場**：持有滿 20 個交易日

In [1]:
# =============================================================
# 1. 設定
# =============================================================
import os, glob, json
import numpy as np
import pandas as pd
from datetime import datetime

DAILY_DIR   = 'Result/DailyFiles'
OUT_DIR     = 'Result/recommend'
LEDGER_PATH = f'{OUT_DIR}/推薦紀錄.csv'
STATE_PATH  = f'{OUT_DIR}/state.json'
BASE_INFO   = 'BaseInfo/StockBaseData.csv'

SEND_LINE  = True     # False：只印訊息不發送
FORCE_SEND = False    # True：沒有新日期也重發最新狀態

# 離場參數
BREAKOUT_STOP = 0.97  # 價起回測：突破線 × 0.97
MAX_LOSS      = 0.08  # 停損最多 -8%
TRAIL_TRIGGER = 0.15  # 曾獲利 15% 後啟動移動停利
TRAIL_DD      = 0.10  # 自最高收盤回落 10%
MAX_HOLD_DAYS = 20    # 持有滿 20 個交易日出場

LINE_MAX_ROWS = 25    # 每張卡片最多列數

LEDGER_COLS = ['推薦日', '代號', '名稱', '市場', '類型', '推薦日收盤', '停損參考',
               '進場日', '進場價', '停損價', '最高收盤', '持有天數',
               '最新日', '最新收盤', '報酬%', '狀態', '離場日', '離場價', '離場原因']
os.makedirs(OUT_DIR, exist_ok=True)

In [2]:
# =============================================================
# 2. 載入
# =============================================================
def _as_bool(s):
    return s.astype(str).str.strip().str.lower().isin(['true', '1'])


def load_daily(path):
    df = pd.read_excel(path, dtype={'stock_number': str})
    df['stock_number'] = df['stock_number'].str.strip()
    return df.drop_duplicates('stock_number', keep='last').set_index('stock_number')


def yahoo_url(code, market):
    suffix = 'TWO' if market == '上櫃' else 'TW'
    return f'https://tw.stock.yahoo.com/quote/{code}.{suffix}'


base = pd.read_csv(BASE_INFO, dtype=str).drop_duplicates('公司代號').set_index('公司代號')

if os.path.exists(LEDGER_PATH):
    ledger = pd.read_csv(LEDGER_PATH, dtype={'代號': str, '推薦日': str, '進場日': str,
                                             '最新日': str, '離場日': str})
    ledger[['名稱', '市場', '類型']] = ledger[['名稱', '市場', '類型']].fillna('')
else:
    ledger = pd.DataFrame(columns=LEDGER_COLS)

state = json.load(open(STATE_PATH, encoding='utf-8')) if os.path.exists(STATE_PATH) else {}
last_done = state.get('last_date', '')

daily_files = {os.path.basename(f)[:-5]: f for f in glob.glob(f'{DAILY_DIR}/*.xlsx')}
all_dates = sorted(daily_files)
if not last_done and all_dates:
    # 第一次執行：只從最新一天開始，不回補舊推薦
    todo_dates = all_dates[-1:]
else:
    todo_dates = [d for d in all_dates if d > last_done]
print('上次處理到:', last_done or '(無)', '| 本次處理:', todo_dates)

上次處理到: (無) | 本次處理: ['2026-10-22']


In [3]:
# =============================================================
# 3. 持倉更新：補進場價、檢查離場
# =============================================================
def update_positions(ledger, day, d, events):
    for i in ledger.index[ledger['狀態'] == '持有中']:
        code = ledger.at[i, '代號']
        if code not in day.index or d <= ledger.at[i, '推薦日']:
            continue                                   # 停牌/當天剛推薦
        r = day.loc[code]
        close = float(r['收盤價'])

        # 推薦隔日開盤進場，確定停損價
        if pd.isna(ledger.at[i, '進場價']):
            entry = float(r['開盤價'])
            ledger.at[i, '進場日'] = d
            ledger.at[i, '進場價'] = entry
            ref = ledger.at[i, '停損參考']
            floor = entry * (1 - MAX_LOSS)
            ledger.at[i, '停損價'] = round(max(ref, floor) if pd.notna(ref) else floor, 2)
            ledger.at[i, '持有天數'] = 0
            ledger.at[i, '最高收盤'] = close

        entry = float(ledger.at[i, '進場價'])
        ledger.at[i, '持有天數'] = int(ledger.at[i, '持有天數']) + 1
        ledger.at[i, '最高收盤'] = max(float(ledger.at[i, '最高收盤']), close)
        ledger.at[i, '最新日'] = d
        ledger.at[i, '最新收盤'] = close
        ledger.at[i, '報酬%'] = round((close / entry - 1) * 100, 2)

        peak = float(ledger.at[i, '最高收盤'])
        ma20 = pd.to_numeric(r.get('MA_20'), errors='coerce')
        reason = ''
        if close < float(ledger.at[i, '停損價']):
            reason = '停損'
        elif pd.notna(ma20) and close < ma20:
            reason = '跌破MA20'
        elif peak >= entry * (1 + TRAIL_TRIGGER) and close <= peak * (1 - TRAIL_DD):
            reason = '移動停利'
        elif int(ledger.at[i, '持有天數']) >= MAX_HOLD_DAYS:
            reason = '時間出場'

        if reason:
            ledger.at[i, '狀態'] = '已離場'
            ledger.at[i, '離場日'] = d
            ledger.at[i, '離場價'] = close
            ledger.at[i, '離場原因'] = reason
            events['exit'].append(i)
    return ledger

In [4]:
# =============================================================
# 4. 新推薦：當日 推薦進場 全部收進來
# =============================================================
def pick_new(ledger, day, d, events):
    if '推薦進場' not in day.columns:
        print(f'{d}：DailyFiles 沒有「推薦進場」欄位（舊版 _genHtml_cyber 產出），略過新推薦')
        return ledger
    rec = day[_as_bool(day['推薦進場'])].copy()
    holding = set(ledger.loc[ledger['狀態'] == '持有中', '代號'])
    rows = []
    for code, r in rec.sort_values('成交金額', ascending=False).iterrows():
        if code in holding:
            continue                                   # 已持有不重複推薦
        kind = str(r.get('推薦進場_類型', ''))
        close = float(r['收盤價'])
        if '價起回測' in kind:
            ref = pd.to_numeric(r.get('價起_突破線'), errors='coerce') * BREAKOUT_STOP
        else:
            ref = pd.to_numeric(r.get('量起_K低'), errors='coerce')
        info = base.loc[code] if code in base.index else {}
        rows.append({
            '推薦日': d, '代號': code,
            '名稱': info.get('公司簡稱', '') if len(info) else '',
            '市場': info.get('Type', '') if len(info) else '',
            '類型': kind, '推薦日收盤': close,
            '停損參考': round(ref, 2) if pd.notna(ref) else np.nan,
            # 進場前先用推薦日收盤估停損，隔日開盤後再更新
            '停損價': round(max(ref, close * (1 - MAX_LOSS)) if pd.notna(ref) else close * (1 - MAX_LOSS), 2),
            '最新日': d, '最新收盤': close, '狀態': '持有中',
        })
    if rows:
        start = len(ledger)
        ledger = pd.concat([ledger, pd.DataFrame(rows)], ignore_index=True)
        events['new'].extend(range(start, len(ledger)))
    return ledger


events = {'new': [], 'exit': []}
for d in todo_dates:
    day = load_daily(daily_files[d])
    ledger = update_positions(ledger, day, d, events)
    ledger = pick_new(ledger, day, d, events)

ledger = ledger.reindex(columns=LEDGER_COLS)
print('新推薦', len(events['new']), '| 離場', len(events['exit']),
      '| 追蹤中', int((ledger['狀態'] == '持有中').sum()))

新推薦 7 | 離場 0 | 持有中 7


In [5]:
# =============================================================
# 5. 存檔
# =============================================================
ledger.to_csv(LEDGER_PATH, index=False, encoding='utf-8-sig')
if todo_dates:
    state['last_date'] = todo_dates[-1]
    state['updated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    json.dump(state, open(STATE_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)

_closed = ledger[ledger['狀態'] == '已離場']
if len(_closed):
    _ret = (_closed['離場價'].astype(float) / _closed['進場價'].astype(float) - 1) * 100
    print(f'累計已離場 {len(_closed)} 筆｜勝率 {(_ret > 0).mean() * 100:.1f}%｜平均 {_ret.mean():.2f}%')
ledger.tail(10)

,推薦日,代號,名稱,市場,類型,推薦日收盤,停損參考,進場日,進場價,停損價,最高收盤,持有天數,最新日,最新收盤,報酬%,狀態,離場日,離場價,離場原因
0,2026-10-22,1216,統一,上市,價起回測,79.60,77.11,NaN,NaN,77.11,NaN,NaN,2026-10-22,79.60,NaN,持有中,NaN,NaN,NaN
1,2026-10-22,4904,遠傳,上市,價起回測+量縮守穩,115.00,109.61,NaN,NaN,109.61,NaN,NaN,2026-10-22,115.00,NaN,持有中,NaN,NaN,NaN
2,2026-10-22,1402,遠東新,上市,價起回測,28.90,27.84,NaN,NaN,27.84,NaN,NaN,2026-10-22,28.90,NaN,持有中,NaN,NaN,NaN
3,2026-10-22,2480,敦陽科,上市,量起後量縮守穩,186.50,177.50,NaN,NaN,177.50,NaN,NaN,2026-10-22,186.50,NaN,持有中,NaN,NaN,NaN
4,2026-10-22,2851,中再保,上市,量起後量縮守穩,50.50,48.70,NaN,NaN,48.70,NaN,NaN,2026-10-22,50.50,NaN,持有中,NaN,NaN,NaN
5,2026-10-22,1416,廣豐,上市,量起後量縮守穩,13.25,12.85,NaN,NaN,12.85,NaN,NaN,2026-10-22,13.25,NaN,持有中,NaN,NaN,NaN
6,2026-10-22,6934,心誠鎂,上市,價起回測,85.50,78.47,NaN,NaN,78.66,NaN,NaN,2026-10-22,85.50,NaN,持有中,NaN,NaN,NaN


In [ ]:
# =============================================================
# 5b. 網頁資料：依「推薦月份」拆檔，給 recommend.html 讀取
#     Result/recommend/web/index.json      月份清單 + 統計
#     Result/recommend/web/YYYY-MM.json    該月推薦的所有紀錄
# =============================================================
WEB_DIR = f'{OUT_DIR}/web'
os.makedirs(WEB_DIR, exist_ok=True)


def _chart_path(code):
    # 對應 _genHtml_cyber 的 save_plt_to_html 檔名：Html/[代號]簡稱-產業-上市/上櫃.html
    if code not in base.index:
        return ''
    r = base.loc[code]
    ind = r.get('上櫃公司產業類別' if r.get('Type') == '上櫃' else '上市公司產業類別', '')
    return f"Html/[{code}]{r.get('公司簡稱', '')}-{ind}-{r.get('Type', '')}.html".replace('*', '')


def _num(v, nd=2):
    return None if pd.isna(v) else round(float(v), nd)


# 族群 / score：取 Untitled 產出的最新 checkall（score = fundamental_score_all 基本面分數）
_extra = {}
_ck_files = sorted(glob.glob('Result/checkall/*.xlsx'))
if _ck_files:
    _ck = pd.read_excel(_ck_files[-1], dtype={'stock_number': str}, usecols=['stock_number', '族群', 'score'])
    _extra = _ck.drop_duplicates('stock_number').set_index('stock_number').to_dict('index')


def _ck_val(code, col):
    v = _extra.get(code, {}).get(col)
    return None if v is None or pd.isna(v) else v


web = ledger.copy()
web_rows = []
for _, r in web.iterrows():
    web_rows.append({
        '推薦日': r['推薦日'], '代號': r['代號'], '名稱': r['名稱'], '類型': r['類型'],
        '族群': _ck_val(r['代號'], '族群') or '',
        'score': None if _ck_val(r['代號'], 'score') is None else round(float(_ck_val(r['代號'], 'score')), 1),
        '收盤': _num(r['推薦日收盤']), '停損': _num(r['停損價']),
        '進場價': _num(r['進場價']), '最新收盤': _num(r['最新收盤']), '報酬%': _num(r['報酬%']),
        '持有天數': None if pd.isna(r['持有天數']) else int(r['持有天數']),
        '狀態': '推薦中' if r['狀態'] == '持有中' else '離場',
        '離場日': '' if pd.isna(r['離場日']) else r['離場日'],
        '離場原因': '' if pd.isna(r['離場原因']) else r['離場原因'],
        'yahoo': yahoo_url(r['代號'], r['市場']), 'chart': _chart_path(r['代號']),
    })

months = {}
for row in web_rows:
    months.setdefault(row['推薦日'][:7], []).append(row)
for m, rows in months.items():
    rows.sort(key=lambda x: (x['推薦日'], x['代號']), reverse=True)
    json.dump(rows, open(f'{WEB_DIR}/{m}.json', 'w', encoding='utf-8'), ensure_ascii=False, separators=(',', ':'))

index = {
    'updated_at': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'last_date': state.get('last_date', ''),
    'months': [{'month': m, 'count': len(rows),
                'active': sum(x['狀態'] == '推薦中' for x in rows)}
               for m, rows in sorted(months.items(), reverse=True)],
}
json.dump(index, open(f'{WEB_DIR}/index.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('網頁資料：', {m['month']: m['count'] for m in index['months']})

In [6]:
# =============================================================
# 6. LINE：每列點擊開 Yahoo 股市
# =============================================================
def _txt(text, flex=1, **kw):
    return {'type': 'text', 'text': str(text), 'size': 'xs', 'flex': flex, 'wrap': False, **kw}


def _bubble(title, color, header, rows, empty_text):
    body = [_txt(title, weight='bold', size='md', color=color), {'type': 'separator', 'margin': 'md'}]
    if rows:
        body.append({'type': 'box', 'layout': 'horizontal', 'spacing': 'sm', 'margin': 'md',
                     'contents': [_txt(h, f, weight='bold', color='#888888') for h, f in header]})
        shown = rows[:LINE_MAX_ROWS]
        for cells_, url in shown:
            body.append({'type': 'box', 'layout': 'horizontal', 'spacing': 'sm', 'margin': 'sm',
                         'action': {'type': 'uri', 'label': 'Yahoo', 'uri': url},
                         'contents': [_txt(v, f, **kw) for v, f, kw in cells_]})
        if len(rows) > len(shown):
            body.append(_txt(f'…另 {len(rows) - len(shown)} 檔（見推薦紀錄.csv）', color='#888888', margin='md'))
        body.append(_txt('點選列可開啟 Yahoo 股市', color='#aaaaaa', margin='lg', size='xxs'))
    else:
        body.append(_txt(empty_text, color='#888888', margin='md'))
    return {'type': 'bubble', 'size': 'mega',
            'body': {'type': 'box', 'layout': 'vertical', 'contents': body}}


def _name(r):
    return f"{r['代號']} {r['名稱']}"


def _ret_color(v):
    return '#e53935' if v > 0 else ('#43a047' if v < 0 else '#555555')   # 台股：紅漲綠跌


def build_line_message(ledger, events, date_label):
    new = ledger.loc[events['new']]
    exits = ledger.loc[events['exit']]
    hold = ledger[ledger['狀態'] == '持有中'].sort_values('報酬%', ascending=False, na_position='last')

    new_rows = [([(_name(r), 3, {}), (f"{r['推薦日收盤']:g}", 2, {'align': 'end'}),
                  (r['類型'].replace('量起後量縮守穩', '量縮守穩'), 3, {}),
                  (f"{r['停損價']:g}", 2, {'align': 'end'})], yahoo_url(r['代號'], r['市場']))
                for _, r in new.iterrows()]
    exit_rows = [([(_name(r), 3, {}), (r['離場原因'], 3, {}),
                   (f"{int(r['持有天數'])}天", 1, {'align': 'end'}),
                   (f"{r['報酬%']:+.1f}%", 2, {'align': 'end', 'color': _ret_color(r['報酬%'])})],
                  yahoo_url(r['代號'], r['市場']))
                 for _, r in exits.iterrows()]
    hold_rows = []
    for _, r in hold.iterrows():
        ret = r['報酬%']
        ret_txt = '待進場' if pd.isna(r['進場價']) else f'{ret:+.1f}%'
        gap = (r['最新收盤'] / r['停損價'] - 1) * 100
        hold_rows.append(([(_name(r), 3, {}),
                           ('-' if pd.isna(r['持有天數']) else f"{int(r['持有天數'])}天", 1, {'align': 'end'}),
                           (ret_txt, 2, {'align': 'end', 'color': _ret_color(0 if pd.isna(ret) else ret)}),
                           (f'{gap:.1f}%', 2, {'align': 'end'})], yahoo_url(r['代號'], r['市場'])))

    bubbles = [
        _bubble(f'📈 {date_label} 新推薦（{len(new_rows)}）', '#1e88e5',
                [('股票', 3), ('收盤', 2), ('類型', 3), ('停損', 2)], new_rows, '今日無新推薦'),
        _bubble(f'🚪 離場訊號（{len(exit_rows)}）', '#e53935',
                [('股票', 3), ('原因', 3), ('天數', 1), ('報酬', 2)], exit_rows, '今日無離場訊號'),
        _bubble(f'📋 追蹤中（{len(hold_rows)}）', '#555555',
                [('股票', 3), ('天數', 1), ('報酬', 2), ('距停損', 2)], hold_rows, '目前無追蹤'),
    ]
    alt = f'{date_label} 新推薦 {len(new_rows)}｜離場 {len(exit_rows)}｜追蹤 {len(hold_rows)}'
    return alt, {'type': 'carousel', 'contents': bubbles}


date_label = (todo_dates[-1] if todo_dates else last_done)[5:].replace('-', '/')
alt_text, flex = build_line_message(ledger, events, date_label)
print(alt_text, '| Flex 大小', len(json.dumps(flex, ensure_ascii=False).encode('utf-8')), 'bytes')

if not todo_dates and not FORCE_SEND:
    print('沒有新的資料日期，不發送 LINE')
elif not SEND_LINE:
    print('SEND_LINE=False，不發送')
else:
    %run _LineMsg.ipynb
    from linebot.models import FlexSendMessage
    try:
        line_bot_api.push_message(user_id, FlexSendMessage(alt_text=alt_text, contents=flex))
        print('LINE 發送成功')
    except Exception as e:
        print('LINE 發送失敗:', e)

10/22 新推薦 7｜離場 0｜持有 7 | Flex 大小 9965 bytes


C:\Users\user\anaconda3\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
C:\Users\user\AppData\Local\Temp\ipykernel_16668\2513558176.py:5: LineBotSdkDeprecatedIn30: Call to deprecated class LineBotApi. (Use v3 class; linebot.v3.<feature>. See https://github.com/line/line-bot-sdk-python/blob/master/README.rst for more details.) -- Deprecated since version 3.0.0.
  line_bot_api = LineBotApi(LINE_CHANNEL_ACCESS_TOKEN)
C:\Users\user\AppData\Local\Temp\ipykernel_16668\713927044.py:83: LineBotSdkDeprecatedIn30: Call to deprecated method push_message. (Use 'from linebot.v3.messaging import MessagingApi' and 'MessagingApi(...).push_message(...)' instead. See https:

LINE 發送成功
